# Flask

In [2]:
#pip install flask

In [1]:
from flask import Flask

## 1. Минимальное приложение

In [5]:
app = Flask("first_app")

@app.route("/hello")
def hello():
    return "Hello, World!"

In [6]:
app.run()

 * Serving Flask app 'first_app'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Sep/2026 19:22:04] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [02/Sep/2026 19:22:12] "GET /hello HTTP/1.1" 200 -


In [7]:
client = app.test_client()

In [12]:
response = client.get("/hello")
print(response.status)
print(response.text)

200 OK
Hello, World!


## 2. Маршрутизация (routing)

In [13]:
app = Flask(__name__)

@app.route('/')
def index():
    return 'Index Page'

@app.route('/hello')
def hello():
    return 'Hello, World'

In [14]:
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Sep/2026 19:27:39] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [02/Sep/2026 19:27:48] "GET /hello HTTP/1.1" 200 -


In [15]:
client = app.test_client()

In [16]:
response = client.get("/")
print(response.status)
print(response.text)

200 OK
Index Page


In [17]:
response = client.get("/hello")
print(response.status)
print(response.text)

200 OK
Hello, World


In [18]:
response = client.get("/client")
print(response.status)
print(response.text)

404 NOT FOUND
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



## 3. Безопасная передача параметров: Escape

In [23]:
from flask import request

app = Flask(__name__)

@app.route("/hello")
def hello():
    name = request.args.get("name", "Flask")
    return f"Hello, {name}!"

In [24]:
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Sep/2026 19:36:26] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [02/Sep/2026 19:40:51] "GET /hello?name=<script>alert("bad")</script> HTTP/1.1" 200 -


In [25]:
from markupsafe import escape

app = Flask(__name__)

@app.route("/hello")
def hello():
    name = request.args.get("name", "Flask")
    return f"Hello, {escape(name)}!"

In [26]:
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Sep/2026 19:42:19] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [02/Sep/2026 19:42:20] "GET /hello?name=<script>alert("bad")</script> HTTP/1.1" 200 -


## 4. Передача параметров (path)

In [87]:
app = Flask(__name__)

@app.get("/users/<int:user_id>")
def get_user(user_id):
    return {
        "user_id": user_id,
        "message": f"Loaded user: {user_id}"
    }

In [20]:
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [02/Sep/2026 19:32:19] "GET /users/1 HTTP/1.1" 200 -
127.0.0.1 - - [02/Sep/2026 19:32:41] "GET /users/2 HTTP/1.1" 200 -
127.0.0.1 - - [02/Sep/2026 19:32:44] "GET /users/1 HTTP/1.1" 200 -
127.0.0.1 - - [02/Sep/2026 19:32:45] "GET /users/1000 HTTP/1.1" 200 -


In [88]:
client = app.test_client()

In [89]:
response = client.get("/users/1")
print(response.status)
print(response.text)

200 OK
{"message":"Loaded user: 1","user_id":1}



In [91]:
response = client.get("/users/2")
print(response.status)
print(response.text)

200 OK
{"message":"Loaded user: 2","user_id":2}



In [90]:
response = client.get("/users/abc")
print(response.status)
print(response.text)

404 NOT FOUND
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



## 5. Передача параметров (query)

In [27]:
app = Flask(__name__)

@app.get("/users")
def users():
    age = request.args.get("age")
    city = request.args.get("city")
    return {
        "age" : age,
        "city": city
    }

In [28]:
client = app.test_client()

In [31]:
response = client.get("/users?age=30&city=Moscow")
print(response.status)
print(response.text)

200 OK
{"age":"30","city":"Moscow"}



## 6. Обработка ошибок

In [32]:
users = {
    1: {"name": "Alice"},
    2: {"name": "Bob"},
    3: {"name": "Charlie"}
}

app = Flask(__name__)

@app.get("/users/<int:user_id>")
def get_user(user_id):
    if user_id not in users:
        return {
            "error": "User not found"
        }, 404

    return users[user_id]

In [33]:
client = app.test_client()

In [34]:
response = client.get("/users/1")
print(response.status)
print(response.text)

200 OK
{"name":"Alice"}



In [35]:
response = client.get("/users/3")
print(response.status)
print(response.text)

200 OK
{"name":"Charlie"}



In [36]:
response = client.get("/users/6")
print(response.status)
print(response.text)

404 NOT FOUND
{"error":"User not found"}



Другой вариант обработки ошибок

In [46]:
from flask import abort

app = Flask(__name__)

@app.get("/users/<int:user_id>")
def get_user(user_id):

    if user_id not in users:
        abort(404)

    return users[user_id]

In [47]:
client = app.test_client()

In [48]:
response = client.get("/users/1")
print(response.status)
print(response.text)

200 OK
{"name":"Alice"}



In [49]:
response = client.get("/users/10")
print(response.status)
print(response.text)

404 NOT FOUND
<!doctype html>
<html lang=en>
<title>404 Not Found</title>
<h1>Not Found</h1>
<p>The requested URL was not found on the server. If you entered the URL manually please check your spelling and try again.</p>



## 7. Blueprints

In [51]:
from flask import Flask, Blueprint

In [52]:
app = Flask(__name__)

users_bp = Blueprint(
    "users",
    __name__,
    url_prefix="/users"
)

products_bp = Blueprint(
    "products",
    __name__,
    url_prefix="/products"
)

ml_bp = Blueprint(
    "ml",
    __name__,
    url_prefix="/ml"
)

@users_bp.get("/<int:user_id>")
def get_user(user_id):
    return {
        "user_id": user_id
    }

@products_bp.get("/<int:product_id>")
def get_product(product_id):
    return {
        "product_id": product_id
    }

@ml_bp.post("/predict")
def predict():
    data = request.get_json()
    return {
        "prediction": 0.73,
        "features": data
    }

app.register_blueprint(users_bp)
app.register_blueprint(products_bp)
app.register_blueprint(ml_bp)

In [53]:
client = app.test_client()

In [54]:
response = client.get("/users/10")
print(response.status_code)
print(response.json)

200
{'user_id': 10}


In [55]:
response = client.get("/products/42")
print(response.status_code)
print(response.json)

200
{'product_id': 42}


In [56]:
response = client.post(
    "/ml/predict",
    json={
        "age": 35,
        "income": 100_000
    }
)
print(response.status_code)
print(response.json)

200
{'features': {'age': 35, 'income': 100000}, 'prediction': 0.73}


## 8. Пример: передача атрибутов в модель

In [57]:
from flask import (
    Flask,
    Blueprint,
    request,
    current_app
)

app = Flask(__name__)

app.config["MODEL_NAME"] = "Customer Churn Model"
app.config["MODEL_VERSION"] = "1.0"

class SimpleModel:
    def predict(self, age, income, credit_score):
        score = (
            0.001 * income
            + 0.01 * credit_score
            - 0.1 * age
        )
        return score

model = SimpleModel()

users_bp = Blueprint(
    "users",
    __name__,
    url_prefix="/users"
)

users = {
    1: {"name": "Alice"},
    2: {"name": "Bob"},
    3: {"name": "Charlie"}
}

@users_bp.get("/<int:user_id>")
def get_user(user_id):
    if user_id not in users:
        return {
            "error": "User not found"
        }, 404
    return users[user_id]

ml_bp = Blueprint(
    "ml",
    __name__,
    url_prefix="/ml"
)

@ml_bp.post("/predict")
def predict():
    data = request.get_json()
    required_fields = [
        "age",
        "income",
        "credit_score"
    ]

    for field in required_fields:
        if field not in data:
            return {
                "error": f"Missing field: {field}"
            }, 400

    prediction = model.predict(
        age=data["age"],
        income=data["income"],
        credit_score=data["credit_score"]
    )

    return {
        "model": current_app.config["MODEL_NAME"],
        "version": current_app.config["MODEL_VERSION"],
        "prediction": prediction
    }

@app.get("/health")
def health():
    return {
        "status": "ok"
    }

app.register_blueprint(users_bp)
app.register_blueprint(ml_bp)

In [58]:
client = app.test_client()

In [59]:
response = client.get("/health")

print(response.status_code)
print(response.json)

200
{'status': 'ok'}


In [60]:
response = client.get("/users/1")

print(response.status_code)
print(response.json)

200
{'name': 'Alice'}


In [61]:
response = client.get("/users/10")

print(response.status_code)
print(response.json)

404
{'error': 'User not found'}


In [63]:
response = client.post(
    "/ml/predict",
    json={
        "age": 35,
        "income": 100000,
        "credit_score": 720
    }
)

print(response.status_code)
print(response.json)

200
{'model': 'Customer Churn Model', 'prediction': 103.7, 'version': '1.0'}


In [64]:
0.001 * 100000 + 0.01 * 720 - 0.1 * 35

103.7

In [65]:
response = client.post(
    "/ml/predict",
    json={
        "age": 35,
        "income": 100_000
    }
)

print(response.status_code)
print(response.json)

400
{'error': 'Missing field: credit_score'}


## 9. Пример: поиск данных в БД PostreSQL

In [66]:
from flask import Flask, jsonify
import psycopg2
import psycopg2.extras

app = Flask(__name__)

def get_connection():
    return psycopg2.connect(
        dbname="dvdrental", user="postgres",
        password="123", host="localhost"
    )

@app.route("/customers/<int:customer_id>")
def get_customer(customer_id):
    connection = get_connection()
    cursor = connection.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    cursor.execute(
        "SELECT customer_id, first_name, last_name, email "
        "FROM customer WHERE customer_id = %s",
        (customer_id,)
    )
    row = cursor.fetchone()
    cursor.close()
    connection.close()

    if row is None:
        return jsonify({"error": "customer not found"}), 404
    return jsonify(dict(row))

In [67]:
client = app.test_client()

In [79]:
response = client.get("/customers/6090")

print(response.status_code)
print(response.json)

404
{'error': 'customer not found'}


# Fast API

## 1. Минимальное приложение

In [30]:
#pip install fastapi

In [93]:
from fastapi import FastAPI, HTTPException, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel

In [94]:
app = FastAPI()

client = TestClient(app)

@app.get("/")
def hello():
    return {
        "message": "Hello, World!"
    }

In [95]:
response = client.get("/")
print(response.status_code)
print(response.json())

200
{'message': 'Hello, World!'}


## 2. Передача параметров (path)

In [96]:
@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {
        "user_id": user_id,
        "message": f"User {user_id}"
    }

In [97]:
response = client.get("/users/123")

print(response.status_code)
print(response.json())

200
{'user_id': 123, 'message': 'User 123'}


In [98]:
response = client.get("/users/abc")

print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'int_parsing', 'loc': ['path', 'user_id'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'abc'}]}


## 3. Передача параметров (query)

In [99]:
@app.get("/users")
def users(age: int, city: str):
    return {
        "age": age,
        "city": city
    }

In [100]:
response = client.get(
    "/users?age=30&city=Amsterdam"
)
print(response.status_code)
print(response.json())

200
{'age': 30, 'city': 'Amsterdam'}


In [101]:
response = client.get(
    "/users?age=30"
)
print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'missing', 'loc': ['query', 'city'], 'msg': 'Field required', 'input': None}]}


In [102]:
@app.get("/customers")
def customers(
    age: int,
    city: str | None = None
):

    return {
        "age": age,
        "city": city
    }

In [103]:
response = client.get("/customers?age=30")

print(response.json())

{'age': 30, 'city': None}


## 4. Валидация параметров

In [104]:
@app.get("/products")
def products(
    limit: int = Query(
            default=10,
            ge=1,
            le=100
        )
    ):
    return {
        "limit": limit
    }

In [106]:
response = client.get("/products")
print(response.status_code)
print(response.json())

200
{'limit': 10}


In [105]:
response = client.get("/products?limit=50")
print(response.status_code)
print(response.json())

200
{'limit': 50}


In [107]:
response = client.get("/products?limit=500")
print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'less_than_equal', 'loc': ['query', 'limit'], 'msg': 'Input should be less than or equal to 100', 'input': '500', 'ctx': {'le': 100}}]}


## 5. JSON request body

In [109]:
from pydantic import BaseModel

class PredictionRequest(BaseModel):
    age: int
    income: float
    credit_score: int

In [110]:
@app.post("/predict")
def predict(data: PredictionRequest):
    return {
        "age": data.age,
        "income": data.income,
        "credit_score": data.credit_score
    }

In [111]:
response = client.post(
    "/predict",
    json={
        "age": 35,
        "income": 100000,
        "credit_score": 720
    }
)
print(response.status_code)
print(response.json())

200
{'age': 35, 'income': 100000.0, 'credit_score': 720}


In [112]:
response = client.post(
    "/predict",
    json={
        "age": "hello",
        "income": 100000,
        "credit_score": 720
    }
)
print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'int_parsing', 'loc': ['body', 'age'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'hello'}]}


In [114]:
response = client.post(
    "/predict",
    json={
        "age": 35,
        "income": 100000
    }
)
print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'missing', 'loc': ['body', 'credit_score'], 'msg': 'Field required', 'input': {'age': 35, 'income': 100000}}]}


## 6. Валидация параметров с Pydantic

In [116]:
from pydantic import BaseModel, Field

class PredictionRequest(BaseModel):
    age: int = Field(ge=18, le=100)
    income: float = Field(ge=0)
    credit_score: int = Field(ge=300, le=850)

@app.post("/predict/2")
def predict(data: PredictionRequest):

    return {
        "age": data.age,
        "income": data.income,
        "credit_score": data.credit_score
    }

In [118]:
response = client.post(
    "/predict/2",
    json={
        "age": 15,
        "income": 100000,
        "credit_score": 720
    }
)

print(response.status_code)
print(response.json())

422
{'detail': [{'type': 'greater_than_equal', 'loc': ['body', 'age'], 'msg': 'Input should be greater than or equal to 18', 'input': 15, 'ctx': {'ge': 18}}]}


## 7. Документация

In [119]:
schema = app.openapi()

In [120]:
print(schema.keys())

dict_keys(['openapi', 'info', 'paths', 'components'])


In [107]:
print(schema["paths"].keys())

dict_keys(['/', '/users/{user_id}', '/users', '/customers', '/products', '/predict', '/predict/2'])


In [108]:
schema["paths"]["/predict"]

{'post': {'summary': 'Predict',
  'operationId': 'predict_predict_post',
  'requestBody': {'content': {'application/json': {'schema': {'$ref': '#/components/schemas/__main____PredictionRequest__2'}}},
   'required': True},
  'responses': {'200': {'description': 'Successful Response',
    'content': {'application/json': {'schema': {}}}},
   '422': {'description': 'Validation Error',
    'content': {'application/json': {'schema': {'$ref': '#/components/schemas/HTTPValidationError'}}}}}}}